# DiffusionGemma stage-1 reproduction (small scale)
Runtime -> Change runtime type -> **A100 GPU**. Run cells top to bottom.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Upload dgemma-colab.zip (built by make_notebook.py)
!rm -f dgemma-colab*.zip
from google.colab import files
files.upload()
!unzip -o -q dgemma-colab.zip && pip install -q -U transformers datasets

In [ ]:
!python -m pytest -q test_dgemma.py

## 1. Warm start from the pretrained AR model (the paper's setting)

In [ ]:
# skipped if warm.pt from the previous run is still on this runtime
import os
if not os.path.exists('warm.pt'):
    !python train.py --init pretrained --steps 3000 --out warm

## 2. Sweep the maximum number of denoising steps (paper Fig. 9)

In [ ]:
!python evaluate.py --ckpt warm.pt --entropy-bound 0.1 --max-steps 96
!python evaluate.py --ckpt warm.pt --entropy-bound 0.1 --max-steps 192
!python evaluate.py --ckpt warm.pt --entropy-bound 5.0 --max-steps 192

In [ ]:
# keep results
!zip -q results.zip *.json *.jsonl
files.download('results.zip')